In [ ]:
# Import libraries
import requests
from dotenv import load_dotenv
import os
import pandas as pd

In [ ]:
# Read sample csv
df = pd.read_csv("1k_sub_sample.csv")
df.info()
df.head()

In [ ]:
# Confirming the data
df[df['ctor'] > 65].head()

In [ ]:
# Loading .env file
load_dotenv()
API_KEY = os.getenv("MAILERLITE_API_KEY")

In [ ]:
# Set up headers
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

Upon doing a test run for a single subscriber to inspect the activity log response structure, it is revealed that each record in 'data' is a separate event (campaign_send, email_open, link_click, etc.). The 'id' field is the activity log entry ID, different id per event.

In [ ]:
sample_id = df['id']
activity_log = []

for id in sample_id:

    page = 1
    while True:

        response = requests.get(
            url = f"https://connect.mailerlite.com/api/subscribers/{id}/activity-log",
            headers = headers,
            params = {
                "limit":100,
                "page":page
            }    
        )
        
        data = response.json()

        for i in data['data']:
            i['subscriber_id'] = id
        activity_log.extend(data['data'])
        if len(data['data']) < 100:
            break
        page += 1


In [ ]:
activity_log

In [ ]:
activity_df = pd.DataFrame(activity_log)
activity_df.info()
activity_df.head()

In [ ]:
# View the full event of a row
activity_log[1241]

In [ ]:
# View a sample of the properties column
# Looking at the various subscriber's activity log
activity_log[8000]['properties']

In [ ]:
# Save to CSV
activity_df.to_csv("activity_log_raw.csv", index=False)